# ClimaCity Paris -- Session 5
## Machine Learning avec MLlib, optimisation et bilan d'architecture

**Module** : Traitement de données massives avec Apache Spark et PySpark  
**Prérequis** : Sessions 2–3 terminées — la table Delta
`data/output/delta/disponibilite` doit être présente (années 2020–2022 typiquement).

**Dépendances supplémentaires** : `folium`, `mlflow` (`pip install -r requirements.txt`).

---

Ce notebook couvre le Jour 3 du projet ClimaCity Paris.

- **Partie 1** : Machine Learning distribué avec MLlib.
  Construction de features, clustering des stations (K-Means), modèle de
  prédiction du taux d'occupation (GBTRegressor), validation croisée,
  et suivi des expériences avec MLflow.

> **Convention** : les cellules `# [EXERCICE]` contiennent une consigne.  
> Les cellules `# [CORRECTION]` proposent une solution.


---
## Section 0 -- Configuration


In [1]:
import os
import sys
import platform
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Mac Apple Silicon : Java arm64 + workers PySpark arm64 (identique Sessions 3–4)
if platform.system() == "Darwin" and platform.machine() == "arm64":
    chemins_jdk_possibles = [
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
    ]
    chemin_java_home = next(
        (
            chemin_jdk
            for chemin_jdk in chemins_jdk_possibles
            if (chemin_jdk / "bin" / "java").exists()
        ),
        None,
    )
    if chemin_java_home:
        os.environ["JAVA_HOME"] = str(chemin_java_home)
        os.environ["PATH"] = f"{chemin_java_home / 'bin'}:{os.environ.get('PATH', '')}"
    chemin_python = Path(sys.executable)
    script_python_arm64 = chemin_python.parent / "python-arm64"
    if not script_python_arm64.exists():
        script_python_arm64.write_text(
            f'#!/bin/bash\nexec arch -arm64 "{chemin_python}" "$@"\n',
            encoding="utf-8",
        )
        script_python_arm64.chmod(0o755)
    os.environ["PYSPARK_PYTHON"] = str(script_python_arm64)
    os.environ["PYSPARK_DRIVER_PYTHON"] = str(chemin_python)

# Chemins projet ("data" à la racine, ou "../data" si cwd différent)
chemins_data_possibles = [Path("data"), Path("../data")]
DATA_DIR = next(
    (
        chemin_data
        for chemin_data in chemins_data_possibles
        if (chemin_data / "velib").exists()
    ),
    chemins_data_possibles[0],
)
OUTPUT_DIR = DATA_DIR / "output"
DELTA_DISPONIBLE = OUTPUT_DIR / "delta" / "disponibilite"
STATIONS_CSV = DATA_DIR / "velib" / "stations_info.csv"
MODELS_DIR = OUTPUT_DIR / "models"
MLFLOW_DIR = OUTPUT_DIR / "mlruns"

assert DELTA_DISPONIBLE.exists(), (
    f"Table Delta manquante : {DELTA_DISPONIBLE.resolve()}\n"
    "Exécutez les Sessions 2–3 avant ce notebook."
)
assert STATIONS_CSV.exists(), f"Fichier manquant : {STATIONS_CSV.resolve()}"

for dossier in [MODELS_DIR, MLFLOW_DIR]:
    dossier.mkdir(parents=True, exist_ok=True)

APP_NAME = "ClimaCity-Paris-Jour3"
SHUFFLE_PARTS = 8
SEED = 42

print(f"[OK] DATA_DIR         : {DATA_DIR.resolve()}")
print(f"[OK] DELTA_DISPONIBLE : {DELTA_DISPONIBLE.resolve()}")
print(f"[OK] STATIONS_CSV     : {STATIONS_CSV.resolve()}")


[OK] DATA_DIR         : /Users/romain/Desktop/SparkVelib/data
[OK] DELTA_DISPONIBLE : /Users/romain/Desktop/SparkVelib/data/output/delta/disponibilite
[OK] STATIONS_CSV     : /Users/romain/Desktop/SparkVelib/data/velib/stations_info.csv


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

_builder = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTS))
    .config("spark.driver.memory", "6g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(_builder).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} — Delta Lake activé — http://localhost:4040")


26/07/23 09:15:37 WARN Utils: Your hostname, MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.10.179.47 instead (on interface en0)
26/07/23 09:15:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/romain/.ivy2/cache
The jars for the packages stored in: /Users/romain/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5ffc4735-e4e7-4cba-8628-99aad866c0ef;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 73ms :: artifacts dl 2ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|    

:: loading settings :: url = jar:file:/Users/romain/Desktop/SparkVelib/.venv-spark/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/07/23 09:15:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.9 — Delta Lake activé — http://localhost:4040


In [3]:
# Chargement de la table Delta consolidée (Sessions 2–3)
df = (
    spark.read.format("delta")
    .load(str(DELTA_DISPONIBLE.resolve()))
)
df.cache()
df.count()  # force le cache

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()


26/07/23 09:15:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Table consolidée : 690,947 lignes  |  20 colonnes
root
 |-- station_id: long (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- code_arr: integer (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- horodatage: string (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- statut: string (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- est_weekend: boolean (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidite_pct: integer (nullable = true)
 |-- vent_kmh: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- est_pluie: boolean (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)



---
# PARTIE 1 -- Apprentissage automatique avec MLlib

## 1.1 Les trois abstractions fondamentales de MLlib

MLlib repose sur trois interfaces qui permettent de construire des pipelines
ML reproductibles et composables.

### Transformer

Un `Transformer` prend un DataFrame en entrée et retourne un DataFrame enrichi.
Il implémente la méthode `transform(df)`.  
Exemples : `VectorAssembler`, `StandardScaler` (après fit), `Binarizer`.

### Estimator

Un `Estimator` est un algorithme qui s'entraîne sur des données.
Il implémente la méthode `fit(df)` et retourne un `Transformer` (le modèle ajusté).  
Exemples : `GBTRegressor`, `KMeans`, `StandardScaler` (avant fit).

### Pipeline

Un `Pipeline` enchaîne une liste ordonnée de `Transformer` et d'`Estimator`.
Lorsqu'on appelle `pipeline.fit(df)`, chaque étape est exécutée en séquence.
Le résultat est un `PipelineModel`, qui lui-même est un `Transformer`.

```
DataFrame d'entrée
      │
      ▼
  Étape 1 : VectorAssembler  (Transformer)  → features brutes
      │
      ▼
  Étape 2 : StandardScaler   (Estimator)    → calcule mean/std sur train
      │         ↓ fit → StandardScalerModel (Transformer)
      ▼
  Étape 3 : GBTRegressor     (Estimator)    → entraîne le modèle
                ↓ fit → GBTRegressionModel  (Transformer)
      │
      ▼
DataFrame de sortie (avec colonne "prediction")
```

La clé du Pipeline : on appelle `fit()` **une seule fois** sur les données
d'entraînement. Le `PipelineModel` résultant peut être appliqué sur les données
de test **sans risque de fuite d'information** entre train et test.


---
## 1.2 Construction des features

Un bon modèle commence par de bonnes features. Nous allons construire
un vecteur de features à partir des informations disponibles :
contexte temporel, conditions météorologiques, et historique récent de la station.


In [4]:
from pyspark.sql.functions import (
    col, sin, cos, lit, lag, lead, coalesce,
    avg as spark_avg,
    round as spark_round,
)
from pyspark.sql.window import Window
import math

# ── 1. Features temporelles cycliques ────────────────────────────────────────
# L'heure 23 et l'heure 0 sont proches dans le temps mais éloignées en valeur
# brute. On encode l'heure, le jour de semaine et le mois sur un cercle (sin/cos).

PI = math.pi

df_feat = (
    df
    .withColumn("heure_sin", sin(lit(2 * PI) * col("heure") / lit(24)))
    .withColumn("heure_cos", cos(lit(2 * PI) * col("heure") / lit(24)))
    .withColumn("jour_sem_sin", sin(lit(2 * PI) * col("jour_sem") / lit(7)))
    .withColumn("jour_sem_cos", cos(lit(2 * PI) * col("jour_sem") / lit(7)))
    .withColumn("mois_sin", sin(lit(2 * PI) * col("mois") / lit(12)))
    .withColumn("mois_cos", cos(lit(2 * PI) * col("mois") / lit(12)))
    .withColumn("est_weekend_int", col("est_weekend").cast("int"))
)

# ── 2. Features météo (imputation + normalisation simple) ────────────────────
# coalesce remplace les NULL par des constantes raisonnables (climat parisien).
# La normalisation ramène les grandeurs dans des ordres de grandeur comparables ;
# le StandardScaler du pipeline fera le centrage/réduction final.

df_feat = (
    df_feat
    .withColumn(
        "temp_norm",
        coalesce(col("temperature_c"), lit(12.0)) / lit(40.0),
    )
    .withColumn(
        "humidite_norm",
        coalesce(col("humidite_pct").cast("double"), lit(70.0)) / lit(100.0),
    )
    .withColumn(
        "vent_norm",
        coalesce(col("vent_kmh"), lit(10.0)) / lit(50.0),
    )
    .withColumn(
        "precip_norm",
        coalesce(col("precipitation_mm"), lit(0.0)) / lit(10.0),
    )
    .withColumn(
        "est_pluie_int",
        coalesce(col("est_pluie").cast("int"), lit(0)),
    )
)

# ── 3. Features de lag (historique récent de la station) ─────────────────────
fenetre_station = (
    Window
    .partitionBy("station_id")
    .orderBy("horodatage")
)
fenetre_moy_4 = fenetre_station.rowsBetween(-4, -1)

df_feat = (
    df_feat
    .withColumn("taux_lag1", lag("taux_occupation", 1).over(fenetre_station))
    .withColumn("taux_lag4", lag("taux_occupation", 4).over(fenetre_station))
    .withColumn("taux_moy_4", spark_avg("taux_occupation").over(fenetre_moy_4))
)

# ── 4. Variable cible : taux_occupation à t+4 (≈ 1 heure dans le futur) ──────
df_feat = df_feat.withColumn(
    "cible",
    lead("taux_occupation", 4).over(fenetre_station),
)

# ── 5. Suppression des lignes incomplètes (lag/lead génère des nulls) ──────────
FEATURES = [
    "heure_sin", "heure_cos",
    "jour_sem_sin", "jour_sem_cos",
    "mois_sin", "mois_cos",
    "est_weekend_int",
    "temp_norm", "humidite_norm", "vent_norm", "precip_norm", "est_pluie_int",
    "taux_lag1", "taux_lag4", "taux_moy_4",
]

df_ml = df_feat.dropna(subset=FEATURES + ["cible"])

print(f"Lignes disponibles pour le ML : {df_ml.count():,}")
print(f"Features ({len(FEATURES)}) : {FEATURES}")


Lignes disponibles pour le ML : 681,456
Features (15) : ['heure_sin', 'heure_cos', 'jour_sem_sin', 'jour_sem_cos', 'mois_sin', 'mois_cos', 'est_weekend_int', 'temp_norm', 'humidite_norm', 'vent_norm', 'precip_norm', 'est_pluie_int', 'taux_lag1', 'taux_lag4', 'taux_moy_4']


## 1.3 Séparation du jeu de données

Créer un jeu de données pour l'entraînement et un jeu de données pour la validation/test

In [5]:
# ── 6. Split train / test ──────────────────────────────────────────────────────
# IMPORTANT : on split dans le temps, pas aléatoirement.
# Un split aléatoire sur des séries temporelles entraîne une fuite d'information :
# le modèle voit des données du futur pendant l'entraînement.

annees = sorted(
    row.annee for row in df_ml.select("annee").distinct().collect()
    if row.annee is not None
)
print(f"Années disponibles : {annees}")

if 2022 in annees and 2023 in annees:
    df_train = df_ml.filter(col("annee") == 2022)
    df_test = df_ml.filter(col("annee") == 2023)
    libelle_train, libelle_test = "2022", "2023"
elif len(annees) >= 2:
    # Dernière année = test, années précédentes = train
    annee_test = annees[-1]
    df_train = df_ml.filter(col("annee") < annee_test)
    df_test = df_ml.filter(col("annee") == annee_test)
    libelle_train = "+".join(str(a) for a in annees[:-1])
    libelle_test = str(annee_test)
else:
    # Une seule année : split temporel 85 / 15
    fenetre_ordre = Window.orderBy("horodatage")
    df_ranked = df_ml.withColumn(
        "rang_temporel",
        F.percent_rank().over(fenetre_ordre),
    )
    df_train = df_ranked.filter(col("rang_temporel") < 0.85).drop("rang_temporel")
    df_test = df_ranked.filter(col("rang_temporel") >= 0.85).drop("rang_temporel")
    libelle_train, libelle_test = "85% début", "15% fin"

n_train = df_train.count()
n_test = df_test.count()
n_total = df_ml.count()

print(f"Train ({libelle_train}) : {n_train:,} lignes")
print(f"Test  ({libelle_test}) : {n_test:,} lignes")
print(f"Ratio        : {n_train / n_total:.1%} / {n_test / n_total:.1%}")

# Mise en cache des deux splits -- ils seront accédés plusieurs fois
df_train.cache()
df_train.count()
df_test.cache()
df_test.count()


Années disponibles : [2020, 2021, 2022]
Train (2020+2021) : 681,410 lignes
Test  (2022) : 46 lignes
Ratio        : 100.0% / 0.0%


46

## 1.4 Clustering des stations avec K-Means

Avant de prédire la disponibilité individuelle de chaque station, nous allons
**regrouper les stations par profil comportemental** : certaines sont saturées
le matin (quartiers résidentiels qui "envoient" des cyclistes vers le centre),
d'autres le soir, d'autres ont un usage homogène sur la journée.

Ce clustering a deux utilités :
1. Comprendre la géographie fonctionnelle du réseau.
2. Ajouter l'identifiant de cluster comme feature dans le modèle de régression.

### Construction du profil de station

Le vecteur de profil d'une station est son **taux d'occupation moyen à chaque
heure de la journée** -- un vecteur de dimension 24.


In [15]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
# ── Profil horaire moyen par station ─────────────────────────────────────────
# Pivot : une ligne par station, une colonne par heure

df_profil = (
    df_ml
    .groupBy("station_id")
    .pivot("heure", list(range(24)))
    .agg(spark_round(spark_avg("taux_occupation"), 4))
)

# Renommage des colonnes pivot (0 -> h00, 1 -> h01, ...)
for h in range(24):
    df_profil = df_profil.withColumnRenamed(str(h), f"h{h:02d}")

colonnes_profil = [f"h{h:02d}" for h in range(24)]
df_profil = df_profil.dropna(subset=colonnes_profil)

print(f"Stations avec profil complet : {df_profil.count()}")
df_profil.show(5, truncate=True)


ModuleNotFoundError: No module named 'distutils'

In [ ]:
# ── Méthode du coude : inertie (WSSSE) en fonction de k ──────────────────────
# On entraîne un K-Means pour k = 2..8 et on trace l'inertie.
# Le "coude" dans la courbe indique le bon compromis.

# VectorAssembler : colonnes h00..h23 → un seul vecteur
assembler_profil = VectorAssembler(
    inputCols=colonnes_profil,
    outputCol="features_profil",
)

# StandardScaler : centrage + réduction (moyenne 0, écart-type 1)
scaler_profil = StandardScaler(
    inputCol="features_profil",
    outputCol="features_scaled",
    withStd=True,
    withMean=True,
)

inertie_par_k = {}

for k in range(2, 9):
    kmeans = KMeans(
        featuresCol="features_scaled",
        predictionCol="cluster",
        k=k,
        seed=SEED,
        maxIter=30,
    )
    pipeline_km = Pipeline(stages=[assembler_profil, scaler_profil, kmeans])
    model_km = pipeline_km.fit(df_profil)

    # WSSSE = Within Set Sum of Squared Errors (inertie intra-cluster)
    wssse = model_km.stages[-1].summary.trainingCost
    inertie_par_k[k] = round(wssse, 2)
    print(f"  k={k}  inertie={wssse:,.1f}")

print("\nTableau récapitulatif :")
for k, w in inertie_par_k.items():
    barre = "=" * int(w / max(inertie_par_k.values()) * 40)
    print(f"  k={k}  {w:>12,.1f}  {barre}")


In [ ]:
# ── Entraînement final avec le k retenu ───────────────────────────────────────
# Choisissez le k correspondant au "coude" observé ci-dessus.
# En pratique sur ce jeu de données, k=4 ou k=5 est souvent pertinent.
K_RETENU = 4

kmeans_final = KMeans(
    featuresCol="features_scaled",
    predictionCol="cluster",
    k=K_RETENU,
    seed=SEED,
    maxIter=50,
)
pipeline_km_final = Pipeline(
    stages=[assembler_profil, scaler_profil, kmeans_final]
)
model_km_final = pipeline_km_final.fit(df_profil)

# DataFrame station_id → cluster
df_clusters = (
    model_km_final.transform(df_profil)
    .select("station_id", "cluster")
)

print(f"Répartition des {df_clusters.count()} stations en {K_RETENU} clusters :")
(
    df_clusters
    .groupBy("cluster")
    .agg(F.count("*").alias("nb_stations"))
    .orderBy("cluster")
    .show()
)


In [ ]:
# ── Interprétation : profil horaire moyen par cluster ────────────────────────

print("Taux d'occupation moyen par cluster et par heure (6h, 9h, 12h, 18h, 22h) :")
(
    df_profil
    .join(df_clusters, on="station_id", how="inner")
    .groupBy("cluster")
    .agg(
        spark_round(spark_avg("h06"), 3).alias("h06"),
        spark_round(spark_avg("h09"), 3).alias("h09"),
        spark_round(spark_avg("h12"), 3).alias("h12"),
        spark_round(spark_avg("h18"), 3).alias("h18"),
        spark_round(spark_avg("h22"), 3).alias("h22"),
        F.count("*").alias("nb_stations"),
    )
    .orderBy("cluster")
    .show()
)


## Folium

Folium s'appuie sur les points forts de l'écosystème Python en termes d'analyse
de données et sur ceux de la bibliothèque JS Leaflet pour l'affichage cartographique.
Manipulez vos données en Python, puis visualisez-les dans une carte via Folium.

### Concepts

Folium facilite la visualisation des données manipulées en Python sur une carte
interactive. Il permet à la fois la liaison de données avec une carte géographique
pour les visualisations choroplèthes, et de passer comme marqueurs des graphismes
riches vectoriels / binaires / HTML.

La bibliothèque dispose d'un certain nombre de jeux de tuiles intégrés
d'OpenStreetMap, Mapbox, etc. Elle prend aussi en charge les jeux de tuiles personnalisés.

Folium prend en charge les superpositions Image, Video, GeoJSON et TopoJSON et
dispose d'un nombre de couches vectorielles intégrées.


In [ ]:
# ── Visualisation sur carte Folium ─────────────────────────────────────────
import pandas as pd

try:
    import folium
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Le module 'folium' est requis pour cette cellule.\n"
        "Installez-le avec : pip install folium"
    ) from exc

PALETTE = [
    "#e74c3c", "#3498db", "#2ecc71", "#f39c12",
    "#9b59b6", "#1abc9c", "#e67e22",
]

# Stations (CSV ; séparateur) jointes aux clusters Spark
df_stations_pd = pd.read_csv(STATIONS_CSV, sep=";")
df_stations_pd = df_stations_pd[["station_id", "lat", "lon", "name"]].copy()

df_clusters_pd = df_clusters.toPandas()
df_carte = df_clusters_pd.merge(df_stations_pd, on="station_id", how="inner")

print(f"Stations cartographiées : {len(df_carte):,}")

carte = folium.Map(
    location=[48.8566, 2.3522],
    zoom_start=12,
    tiles="CartoDB positron",
)

for _, row in df_carte.iterrows():
    couleur = PALETTE[int(row["cluster"]) % len(PALETTE)]
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5,
        color=couleur,
        fill=True,
        fill_color=couleur,
        fill_opacity=0.8,
        popup=folium.Popup(
            f"<b>{row['name']}</b><br>Cluster {int(row['cluster'])}",
            max_width=250,
        ),
        tooltip=f"{row['name']} — cluster {int(row['cluster'])}",
    ).add_to(carte)

# Légende textuelle
legende_html = (
    "<div style='position:fixed;bottom:30px;left:30px;z-index:1000;"
    "background:white;padding:10px;border:1px solid #ccc;font-size:12px'>"
)
for i in range(K_RETENU):
    legende_html += (
        f"<div><span style='display:inline-block;width:14px;height:14px;"
        f"background:{PALETTE[i]};border-radius:50%;margin-right:6px'></span>"
        f"Cluster {i}</div>"
    )
legende_html += "</div>"
carte.get_root().html.add_child(folium.Element(legende_html))

chemin_carte = OUTPUT_DIR / "carte_clusters_velib.html"
carte.save(str(chemin_carte))
print(f"Carte sauvegardée : {chemin_carte.resolve()}")
print("Ouvrez ce fichier dans votre navigateur pour visualiser les clusters.")

carte


In [ ]:
# Ajout du cluster comme feature pour le modèle de régression
df_ml = df_ml.join(df_clusters, on="station_id", how="left").fillna(0, subset=["cluster"])
df_train = df_train.join(df_clusters, on="station_id", how="left").fillna(0, subset=["cluster"])
df_test = df_test.join(df_clusters, on="station_id", how="left").fillna(0, subset=["cluster"])

# Re-cache après jointure (schéma enrichi)
df_train.cache()
df_test.cache()
df_train.count()
df_test.count()

FEATURES = FEATURES + ["cluster"]
print(f"Features après ajout du cluster ({len(FEATURES)}) : {FEATURES}")


---
## 1.5 Modèle de régression : GBTRegressor

Le **Gradient Boosted Tree Regressor** est un algorithme d'ensemble qui construit
séquentiellement des arbres de décision, chacun corrigeant les erreurs du précédent.
C'est l'un des meilleurs algorithmes pour les données tabulaires avec des features
hétérogènes -- exactement notre cas.

MLlib l'implémente de façon distribuée : chaque arbre est construit en parallèle
sur les partitions du DataFrame.


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd

# ── Construction du Pipeline ──────────────────────────────────────────────────

# 1. Vecteur de features brutes
assembler = VectorAssembler(
    inputCols=FEATURES,
    outputCol="features_brutes",
    handleInvalid="skip",
)

# 2. Normalisation centrée-réduite
scaler = StandardScaler(
    inputCol="features_brutes",
    outputCol="features",
    withStd=True,
    withMean=True,
)

# 3. Gradient Boosted Trees
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="cible",
    predictionCol="prediction",
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,
    seed=SEED,
)

# 4. Pipeline complet
pipeline_gbt = Pipeline(stages=[assembler, scaler, gbt])

# ── Entraînement ──────────────────────────────────────────────────────────────
print("Entraînement du GBTRegressor (peut prendre 1-2 minutes)...")
t0 = time.perf_counter()
model_gbt = pipeline_gbt.fit(df_train)
t_fit = time.perf_counter() - t0
print(f"Entraînement terminé en {t_fit:.1f} s")


In [ ]:
# ── Évaluation sur le test set ────────────────────────────────────────────────
evaluator_rmse = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="r2"
)
evaluator_mae = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="mae"
)

df_pred = model_gbt.transform(df_test)

rmse = evaluator_rmse.evaluate(df_pred)
r2 = evaluator_r2.evaluate(df_pred)
mae = evaluator_mae.evaluate(df_pred)

print("Métriques sur le test set :")
print(f"  RMSE (Root Mean Squared Error) : {rmse:.4f}")
print(f"  MAE  (Mean Absolute Error)     : {mae:.4f}")
print(f"  R²   (coefficient de détermination) : {r2:.4f}")
print()
print("Interprétation :")
print(f"  En moyenne, le modèle se trompe de ±{mae:.3f} sur le taux d'occupation (0-1).")
print(f"  Soit ±{mae * 100:.1f} points de pourcentage.")


In [ ]:
# ── Importance des features ────────────────────────────────────────────────────
# Le GBT calcule l'importance de chaque feature (réduction d'impureté moyenne)
import pandas as pd

model_gbt_stage = model_gbt.stages[-1]  # le GBTRegressionModel
importances = model_gbt_stage.featureImportances.toArray()

df_importance = (
    pd.DataFrame({
        "feature": FEATURES,
        "importance": importances,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Importance des features (top 10) :")
print(f"{'Rang':<5} {'Feature':<25} {'Importance':>12}  Barre")
print("-" * 65)
for i, row in df_importance.head(10).iterrows():
    barre = "█" * int(row["importance"] * 200)
    print(f"  {i + 1:<3} {row['feature']:<25} {row['importance']:>12.4f}  {barre}")


In [ ]:
# ── Analyse des erreurs : où le modèle se trompe-t-il le plus ? ───────────────
df_erreurs = (
    df_pred
    .withColumn("erreur_abs", F.abs(col("prediction") - col("cible")))
    .withColumn("erreur_rel",
        F.abs(col("prediction") - col("cible")) / (col("cible") + F.lit(0.01))
    )
)

print("Erreur absolue moyenne par heure de la journée :")
(
    df_erreurs
    .groupBy("heure")
    .agg(
        spark_round(spark_avg("erreur_abs"), 4).alias("mae_horaire"),
        F.count("*").alias("n")
    )
    .orderBy("heure")
    .show(24)
)


In [ ]:
# [EXERCICE]
# Calculez l'erreur absolue moyenne (MAE) par cluster de station.
# Quel cluster est le plus difficile à prédire ? Pourquoi, à votre avis ?
#
# Indice : groupBy("cluster").agg(...)
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
(
    df_erreurs
    .groupBy("cluster")
    .agg(
        spark_round(spark_avg("erreur_abs"), 4).alias("mae_cluster"),
        spark_round(spark_avg("erreur_rel"), 4).alias("mare_cluster"),
        F.count("*").alias("n_predictions"),
    )
    .orderBy(F.desc("mae_cluster"))
    .show()
)


In [ ]:
# [CORRECTION]
print("MAE par cluster de station :")
(
    df_erreurs
    .groupBy("cluster")
    .agg(
        spark_round(spark_avg("erreur_abs"), 4).alias("mae_cluster"),
        spark_round(spark_avg("erreur_rel"), 4).alias("mare_cluster"),
        F.count("*").alias("n_predictions")
    )
    .orderBy(F.desc("mae_cluster"))
    .show()
)
# Les stations de cluster à fort usage (très vides le matin, très pleines le soir)
# sont généralement plus difficiles à prédire car leur dynamique est plus abrupte.


---
## 1.6 Validation croisée et optimisation des hyperparamètres

Nos hyperparamètres actuels (`maxIter=50`, `maxDepth=5`, `stepSize=0.1`)
ont été choisis arbitrairement. Le `CrossValidator` de MLlib permet
d'évaluer systématiquement plusieurs combinaisons et de choisir la meilleure.

> **Attention** : la validation croisée sur des séries temporelles est
> délicate. Le `CrossValidator` de MLlib effectue un K-Fold standard
> (mélanges aléatoires des données), ce qui peut créer de la fuite
> d'information temporelle. Pour un usage production, on préférerait
> une validation en fenêtre glissante (*time series split*).
> Dans le contexte de ce cours, le K-Fold est acceptable pour comprendre
> le mécanisme.


In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# ── Grille d'hyperparamètres ──────────────────────────────────────────────────
param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth,  [3, 5])
    .addGrid(gbt.maxIter,   [30, 50])
    .addGrid(gbt.stepSize,  [0.05, 0.1])
    .build()
)
print(f"Combinaisons à évaluer : {len(param_grid)}")

# ── CrossValidator ────────────────────────────────────────────────────────────
cv = CrossValidator(
    estimator=pipeline_gbt,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_rmse,
    numFolds=3,
    seed=SEED,
    parallelism=2    # évalue 2 combinaisons en parallèle
)

# On entraîne sur un sous-échantillon pour limiter le temps de calcul en cours
df_train_sample = df_train.sample(fraction=0.3, seed=SEED).cache()
df_train_sample.count()
print(f"Sous-échantillon train : {df_train_sample.count():,} lignes")

print("\nValidation croisée en cours (peut prendre 3-5 minutes)...")
t0 = time.perf_counter()
cv_model = cv.fit(df_train_sample)
t_cv = time.perf_counter() - t0
print(f"Terminé en {t_cv:.1f} s")


In [ ]:
import pandas as pd

# ── Résultats de la grille ────────────────────────────────────────────────────
avg_metrics = cv_model.avgMetrics
params_list = [
    {p.name: v for p, v in zip(param_grid[0].keys(), combo.values())}
    for combo in param_grid
]

resultats_cv = pd.DataFrame(params_list)
resultats_cv["rmse_cv"] = avg_metrics
resultats_cv = resultats_cv.sort_values("rmse_cv").reset_index(drop=True)

print("Résultats du CrossValidator (triés par RMSE croissant) :")
print(resultats_cv.to_string(index=False))

# Meilleurs hyperparamètres
meilleur = resultats_cv.iloc[0]
print(f"\nMeilleure combinaison :")
for col_name in ["maxDepth", "maxIter", "stepSize", "rmse_cv"]:
    print(f"  {col_name:<12} : {meilleur[col_name]}")


In [ ]:
# Évaluation du meilleur modèle sur le test set complet
df_pred_best = cv_model.bestModel.transform(df_test)

rmse_best = evaluator_rmse.evaluate(df_pred_best)
r2_best   = evaluator_r2.evaluate(df_pred_best)
mae_best  = evaluator_mae.evaluate(df_pred_best)

print("Comparaison modèle initial vs meilleur modèle CV :")
print(f"{'Métrique':<8}  {'Initial':>12}  {'CV Best':>12}  {'Gain':>10}")
print("-" * 46)
print(f"{'RMSE':<8}  {rmse:>12.4f}  {rmse_best:>12.4f}  {rmse - rmse_best:>+10.4f}")
print(f"{'MAE':<8}  {mae:>12.4f}  {mae_best:>12.4f}  {mae - mae_best:>+10.4f}")
print(f"{'R²':<8}  {r2:>12.4f}  {r2_best:>12.4f}  {r2_best - r2:>+10.4f}")


# 1.7 Suivi des expériences avec MLflow

MLflow est un outil de tracking d'expériences ML : il enregistre
hyperparamètres, métriques, artefacts (modèles, graphiques) et permet
de comparer les runs dans une interface web.

```bash
# Pour ouvrir l'interface MLflow dans un terminal séparé :
mlflow ui --backend-store-uri data/output/mlruns --port 5000
# Puis ouvrir http://localhost:5000 dans le navigateur
```

> **Dépendance** : `pip install mlflow` (déjà listé dans `requirements.txt`).


In [ ]:
import mlflow
import mlflow.spark

# Configuration du répertoire de tracking
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR.resolve()}")
mlflow.set_experiment("ClimaCity-Paris-GBT")

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Expérience          : ClimaCity-Paris-GBT")


In [ ]:
def evaluer_et_logger(
    df_train_fit, df_test_eval,
    max_depth: int, max_iter: int, step_size: float,
    nom_run: str
) -> dict:
    """Entraîne un pipeline GBT, l'évalue et logue les résultats dans MLflow.

    Args:
        df_train_fit  : DataFrame d'entraînement.
        df_test_eval  : DataFrame de test.
        max_depth     : Profondeur maximale des arbres.
        max_iter      : Nombre d'arbres (iterations).
        step_size     : Taux d'apprentissage.
        nom_run       : Nom du run MLflow.

    Returns:
        Dictionnaire avec les métriques calculées sur le test set.
    """
    with mlflow.start_run(run_name=nom_run):
        # ── Paramètres ──
        mlflow.log_params({
            "max_depth" : max_depth,
            "max_iter"  : max_iter,
            "step_size" : step_size,
            "n_features": len(FEATURES),
            "n_train"   : df_train_fit.count(),
            "n_test"    : df_test_eval.count(),
        })

        # ── Entraînement ──
        gbt_run = GBTRegressor(
            featuresCol="features", labelCol="cible",
            predictionCol="prediction",
            maxIter=max_iter, maxDepth=max_depth,
            stepSize=step_size, seed=SEED
        )
        pipeline_run = Pipeline(stages=[assembler, scaler, gbt_run])
        t0    = time.perf_counter()
        model = pipeline_run.fit(df_train_fit)
        t_fit = time.perf_counter() - t0

        # ── Métriques ──
        preds = model.transform(df_test_eval)
        metriques = {
            "rmse"     : evaluator_rmse.evaluate(preds),
            "mae"      : evaluator_mae.evaluate(preds),
            "r2"       : evaluator_r2.evaluate(preds),
            "fit_time" : round(t_fit, 2),
        }
        mlflow.log_metrics(metriques)

        # ── Modèle ──
        mlflow.spark.log_model(model, artifact_path="pipeline_gbt")

        print(f"  [{nom_run}]  RMSE={metriques['rmse']:.4f}  "
              f"R²={metriques['r2']:.4f}  ({t_fit:.1f}s)")
        return metriques

# ── Plusieurs runs pour comparer ──────────────────────────────────────────────
print("Lancement des runs MLflow...")
configs = [
    {"max_depth": 3, "max_iter": 30, "step_size": 0.1,  "nom_run": "shallow-fast"},
    {"max_depth": 5, "max_iter": 50, "step_size": 0.1,  "nom_run": "medium-balanced"},
    {"max_depth": 5, "max_iter": 50, "step_size": 0.05, "nom_run": "medium-slow-lr"},
    {"max_depth": 7, "max_iter": 80, "step_size": 0.05, "nom_run": "deep-thorough"},
]

resultats_mlflow = []
for cfg in configs:
    m = evaluer_et_logger(df_train, df_test, **cfg)
    resultats_mlflow.append({"run": cfg["nom_run"], **m})

print("\nTableau comparatif :")
df_res = pd.DataFrame(resultats_mlflow).sort_values("rmse")
print(df_res.to_string(index=False))


In [ ]:
# ── Chargement du meilleur modèle depuis MLflow ───────────────────────────────
from mlflow.tracking import MlflowClient

client  = MlflowClient()
exp     = client.get_experiment_by_name("ClimaCity-Paris-GBT")
runs    = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.rmse ASC"],
    max_results=1
)
best_run = runs[0]

print(f"Meilleur run :")
print(f"  ID   : {best_run.info.run_id}")
print(f"  Nom  : {best_run.data.tags.get('mlflow.runName', 'N/A')}")
print(f"  RMSE : {best_run.data.metrics['rmse']:.4f}")
print(f"  R²   : {best_run.data.metrics['r2']:.4f}")

# Rechargement du modèle
model_uri     = f"runs:/{best_run.info.run_id}/pipeline_gbt"
model_recharge = mlflow.spark.load_model(model_uri)
print(f"\nModèle rechargé depuis : {model_uri}")

# Vérification : les prédictions sont identiques
df_verif = model_recharge.transform(df_test.limit(100))
rmse_verif = evaluator_rmse.evaluate(df_verif)
print(f"RMSE sur 100 lignes (vérification) : {rmse_verif:.4f}")


In [ ]:
# Sauvegarde locale du meilleur modèle (format natif Spark)
chemin_model_local = MODELS_DIR / "gbt_best"
cv_model.bestModel.write().overwrite().save(str(chemin_model_local))
print(f"Modèle sauvegardé localement : {chemin_model_local}")

# Pour le recharger plus tard :
# from pyspark.ml import PipelineModel
# model_recharge = PipelineModel.load(str(chemin_model_local))
